# Gene ↔ Disease Relation-Wise Merge

Merges Gene–Disease triples from Monarch, DRKG, PharmKG, TARKG, Harmonizome (×7), and
EvoAGE; resolves disease tail names/IDs via DO/MESH (incl. MONDO→MESH), gene head names
via NCBI; deduplicates by `(head, relation, tail)`; and saves the result.

## 0. Configuration

In [3]:
import os
import pandas as pd
import numpy as np
import re

BASE_DIR   = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR   = BASE_DIR + 'processed_data/'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

OUT_PATH   = BASE_DIR + 'processed_data_relation_wise_merge/generalised/GENE_DISEASE/ALL_GENE_DISEASE.csv'

REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'kg_type', 'species'
]

## 1. Build Lookup Dictionaries

### 1a. Gene — NCBI and ENSEMBL

In [14]:
ENS_2NCBI = pd.read_csv(DB_DIR + 'ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv')
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI

NCBI_ALL_GENE = pd.read_csv(DB_DIR + 'ncbi/Homo_sapiens.gene_info', sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict   = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict = dict(zip(NCBI_ALL_GENE['GeneID'], NCBI_ALL_GENE['Symbol']))
NCBI_ALL_Symb_Desc_dict  = dict(zip(NCBI_ALL_GENE['Symbol'], NCBI_ALL_GENE['description']))

NCBI_ALL_GENE_Syn_Dict = dict(zip(NCBI_ALL_GENE['Synonyms'], NCBI_ALL_GENE['Symbol']))
exploded_dict_NCBI_ALL_GENE_Syn_Dict = {}
for k, v in NCBI_ALL_GENE_Syn_Dict.items():
    for alias in k.split('|'):
        exploded_dict_NCBI_ALL_GENE_Syn_Dict[alias.strip()] = v

print(f"NCBI gene table: {len(NCBI_ALL_GENE):,} rows")
print(f"Synonym dict:    {len(exploded_dict_NCBI_ALL_GENE_Syn_Dict):,} entries")

NCBI gene table: 193,505 rows
Synonym dict:    69,564 entries


### 1b. Disease — DO, MedGen, MESH

In [15]:
DO_All_File = pd.read_csv(DB_DIR + 'DO/All_DO_ref_files.csv')
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
print(f"DO dict: {len(DOID_disname_dict):,} entries")

DO dict: 11,804 entries


In [16]:
MedGen_CUID_Source_ID = pd.read_csv(DB_DIR + 'MESH/MeSH/MedGenIDMappings.txt', sep='\t')
MONDO_2_MESH = MedGen_CUID_Source_ID[MedGen_CUID_Source_ID['source_id'].str.contains('MONDO', na=False)]
MONDO_2_MESH_dict = dict(zip(MONDO_2_MESH['source_id'], MONDO_2_MESH['#CUI_or_CN_id']))

MedGen_CUID_Source_ID = MedGen_CUID_Source_ID.drop_duplicates(subset=['source_id', 'source'], keep='first')
MedGen_MeshID_name_dict = dict(zip(MedGen_CUID_Source_ID['source_id'], MedGen_CUID_Source_ID['pref_name']))
print(f"MONDO→MESH: {len(MONDO_2_MESH_dict):,} entries")

MONDO→MESH: 22,703 entries


In [17]:
Mesh2DOID = pd.read_csv(DB_DIR + 'DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv', sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict = dict(zip(Mesh2DOID['xrefs'], Mesh2DOID['id']))

MESH = pd.read_csv(DB_DIR + 'MESH/MESH_Combined.csv')
MESH_dict = dict(zip(MESH['ID'], MESH['Name']))

Mesh_supp = pd.read_csv(DB_DIR + 'MESH/Mesh_Supplementary_Info.csv')
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))
print(f"MESH dict: {len(MESH_dict):,} | MESH supp: {len(Mesh_supp_dict):,}")

rest_mesh = pd.read_csv(DB_DIR + 'MESH/Rest_mesh_collectedfrom_BioGrakn.csv')
rest_mesh_dict = dict(zip(rest_mesh['Tail'], rest_mesh['diseaseName']))


MedGen_CUID_Source_ID = pd.read_csv(DB_DIR + 'MESH/MeSH/MedGenIDMappings.txt',sep = '\t') #,index = None)
MedGen_CUID_Source_ID
MedGen_CUID_Source_ID = MedGen_CUID_Source_ID.drop_duplicates(subset=['source_id','source'], keep='first')
MedGen_CUID_Source_ID
MedGen_MeshID_name_dict = dict(zip(MedGen_CUID_Source_ID['source_id'], MedGen_CUID_Source_ID['pref_name']))
MedGen_MeshID_name_dict
MedGen_CUID_Source_ID

# MESH ID TO DOID
MedGen_CUID_HP_CONVERT = MedGen_CUID_Source_ID[
    MedGen_CUID_Source_ID['source_id'].fillna('').str.startswith('HP:')
]
MedGen_CUID_HP_CONVERT

MedGen_CUID_HP_CONVERT_dict = dict(zip(MedGen_CUID_HP_CONVERT['#CUI_or_CN_id'], MedGen_CUID_HP_CONVERT['source_id']))
MedGen_CUID_HP_CONVERT_dict



print(MedGen_CUID_Source_ID[MedGen_CUID_Source_ID['source_id'] == 'D001932'])
print(MedGen_CUID_Source_ID[MedGen_CUID_Source_ID['source_id'] == 'D003110'])
MedGen_CUID_Source_ID[MedGen_CUID_Source_ID['source_id'] == 'D007022']

MedGen_CUID_Source_ID_name_dict = dict(zip(MedGen_CUID_Source_ID['#CUI_or_CN_id'], MedGen_CUID_Source_ID['pref_name']))
MedGen_CUID_Source_ID_name_dict = {k: v.lower() if isinstance(v, str) else v for k, v in MedGen_CUID_Source_ID_name_dict.items()}
MedGen_CUID_Source_ID_name_dict

MESH dict: 38,520 | MESH supp: 324,150
     #CUI_or_CN_id          pref_name source_id source  Unnamed: 4
3114      C0006118  Neoplasm of brain   D001932   MeSH         NaN
     #CUI_or_CN_id     pref_name source_id source  Unnamed: 4
3265      C0007102  Colon cancer   D003110   MeSH         NaN


{'C0000727': 'acute abdomen',
 'C0000731': 'abdominal distention',
 'C0000734': 'abdominal mass',
 'C0000735': 'neoplasm of abdomen',
 'C0000744': 'abetalipoproteinaemia',
 'C0000765': 'excessive weight gain',
 'C0000768': 'fetal anomaly',
 'C0000772': 'multiple congenital anomalies',
 'C0000774': 'gastrin secretion abnormality',
 'C0000778': 'abo blood group system',
 'C0000790': 'past pregnancy history of pregnancy with abortive outcome',
 'C0000804': 'illegal abortion',
 'C0000809': 'habitual spontaneous abortion',
 'C0000810': 'retained products after miscarriage',
 'C0000814': 'missed abortion',
 'C0000820': 'therapeutic abortion',
 'C0000822': 'miscarriage of tubal ectopic pregnancy',
 'C0000833': 'abscess',
 'C0000729': 'abdominal cramps',
 'C0000880': 'acanthamoeba keratitis',
 'C0000889': 'acanthosis nigricans',
 'C0001075': 'achlorhydria',
 'C0001079': 'achondrogenesis',
 'C0001080': 'achondroplasia',
 'C0001118': 'abnormality of acid-base homeostasis',
 'C0001122': 'acidosis

## 2. Load KG Sources

### 2a. Monarch

Map MONDO tail IDs → MESH, drop unmapped MONDO rows, assign `tail_id_is`.

In [18]:
Monarch_Gene_Disease = pd.read_csv(PROC_DIR + 'Monarch/Human/Human_Gene_Disease_Monarch.csv')
Monarch_Gene_Disease.columns = Monarch_Gene_Disease.columns.str.lower()
Monarch_Gene_Disease['kg_source'] = 'MonarchKG'
Monarch_Gene_Disease.rename(columns={'tail_name': 'tail_detail_name'}, inplace=True)

# MONDO → MESH on tail, then drop unmapped MONDO rows
Monarch_Gene_Disease['tail'] = Monarch_Gene_Disease['tail'].map(MONDO_2_MESH_dict).fillna(Monarch_Gene_Disease['tail'])
Monarch_Gene_Disease = Monarch_Gene_Disease[~Monarch_Gene_Disease['tail'].astype(str).str.startswith('MONDO')]
Monarch_Gene_Disease['tail_id_is'] = np.where(
    Monarch_Gene_Disease['tail'].isna(), None,
    np.where(Monarch_Gene_Disease['tail'].astype(str).str.startswith('DOID'), 'DOID', 'MESH')
)
print(f"Monarch:     {len(Monarch_Gene_Disease):,} rows")
Monarch_Gene_Disease['kg_type'] = 'Generalised'
Monarch_Gene_Disease['kg_source'] = 'MonarchKG'
Monarch_Gene_Disease['species'] = 'Homo species'

Monarch_Gene_Disease

Monarch:     12,461 rows


,head,relation,tail,head_type,relation_type,tail_type,relation_source,kg_source,head_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,head_name,tail_detail_name,head_orignal,species_species,kg_type,species
0,CARD9,Gene_Disease,C1859353,Gene,causes,Disease,infores:omim,MonarchKG,caspase recruitment domain family member 9,Homo sapiens,NaN,HGNC,MESH,CARD9,predisposition to invasive fungal disease due ...,HGNC:16391,Homo sapiens_Homo sapiens,Generalised,Homo species
1,TBC1D7,Gene_Disease,C3806412,Gene,causes,Disease,infores:omim,MonarchKG,TBC1 domain family member 7,Homo sapiens,NaN,HGNC,MESH,TBC1D7,"macrocephaly/megalencephaly syndrome, autosoma...",HGNC:21066,Homo sapiens_Homo sapiens,Generalised,Homo species
2,IFT81,Gene_Disease,DOID:0080295,Gene,causes,Disease,infores:omim,MonarchKG,intraflagellar transport 81,Homo sapiens,NaN,HGNC,DOID,IFT81,short-rib thoracic dysplasia 19 with or withou...,HGNC:14313,Homo sapiens_Homo sapiens,Generalised,Homo species
3,LZTR1,Gene_Disease,DOID:0060588,Gene,causes,Disease,infores:omim,MonarchKG,leucine zipper like post translational regulat...,Homo sapiens,NaN,HGNC,DOID,LZTR1,Noonan syndrome 10,HGNC:6742,Homo sapiens_Homo sapiens,Generalised,Homo species
4,SLC1A1,Gene_Disease,DOID:0070093,Gene,contributes_to,Disease,infores:omim,MonarchKG,solute carrier family 1 member 1,Homo sapiens,NaN,HGNC,DOID,SLC1A1,schizophrenia 18,HGNC:10939,Homo sapiens_Homo sapiens,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12873,THSD1,Gene_Disease,DOID:0060228,Gene,gene_associated_with_condition,Disease,infores:orphanet,MonarchKG,thrombospondin type 1 domain containing 1,Homo sapiens,NaN,HGNC,DOID,THSD1,intracranial berry aneurysm,HGNC:17754,Homo sapiens_Homo sapiens,Generalised,Homo species
12874,TGFBR3,Gene_Disease,DOID:0060228,Gene,gene_associated_with_condition,Disease,infores:orphanet,MonarchKG,transforming growth factor beta receptor 3,Homo sapiens,NaN,HGNC,DOID,TGFBR3,intracranial berry aneurysm,HGNC:11774,Homo sapiens_Homo sapiens,Generalised,Homo species
12875,COL3A1,Gene_Disease,DOID:0060228,Gene,gene_associated_with_condition,Disease,infores:orphanet,MonarchKG,collagen type III alpha 1 chain,Homo sapiens,NaN,HGNC,DOID,COL3A1,intracranial berry aneurysm,HGNC:2201,Homo sapiens_Homo sapiens,Generalised,Homo species
12876,ANGPTL6,Gene_Disease,DOID:0060228,Gene,gene_associated_with_condition,Disease,infores:orphanet,MonarchKG,angiopoietin like 6,Homo sapiens,NaN,HGNC,DOID,ANGPTL6,intracranial berry aneurysm,HGNC:23140,Homo sapiens_Homo sapiens,Generalised,Homo species


### 2b. DRKG

In [19]:
DRKG_Gene_Disease = pd.read_csv(PROC_DIR + 'DRKG/DRKG_Gene_Disease.csv')
DRKG_Gene_Disease.columns = DRKG_Gene_Disease.columns.str.lower()
print(f"DRKG:        {len(DRKG_Gene_Disease):,} rows")
DRKG_Gene_Disease['kg_type'] = 'Generalised'
DRKG_Gene_Disease['kg_source'] = 'DRKG'
DRKG_Gene_Disease['species'] = 'Homo species'
DRKG_Gene_Disease.head(2)

DRKG:        63,439 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id,head_detail_name,head_ens,tail_detail_name,tail_doid_name,head_id_is,tail_id_is,head_orignal,tail_orignal,kg_type,species
0,A1BG,Gene_Disease,D005909,Gene,GNBR::L::Gene:Disease,Disease,DRKG,1,alpha-1-B glycoprotein,ENSG00000121410,Glioblastoma,D005909,NCBI_ID,MESH,Gene::1,Disease::MESH:D005909,Generalised,Homo species
1,NAT2,Gene_Disease,D001172,Gene,GNBR::Y::Gene:Disease,Disease,DRKG,10,N-acetyltransferase 2,ENSG00000156006,"Arthritis, Rheumatoid",D001172,NCBI_ID,MESH,Gene::10,Disease::MESH:D001172,Generalised,Homo species


### 2c. PharmKG

In [20]:
PharmKG_Gene_Disease = pd.read_csv(PROC_DIR + 'PharmKG/PharmKG_Gene_Disease.csv')
PharmKG_Gene_Disease.columns = PharmKG_Gene_Disease.columns.str.lower()
PharmKG_Gene_Disease['tail_detail_name'] = PharmKG_Gene_Disease['tail'].map(DOID_disname_dict)
print(f"PharmKG:     {len(PharmKG_Gene_Disease):,} rows")
PharmKG_Gene_Disease['kg_type'] = 'Generalised'
PharmKG_Gene_Disease['kg_source'] = 'PharmKG'
PharmKG_Gene_Disease['species'] = 'Homo species'
PharmKG_Gene_Disease

PharmKG:     28,273 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,head_id_is,tail_id_is,tail_orignal,pubmed_id,sentence_tokenized,tail_detail_name,kg_type,species
0,CIC,Gene_Disease,DOID:824,gene,J,disease,PharmKG,solute carrier family 25 member 1,NCBI_ID,DOID_ID,periodontitis,2278478.0,'The levels of CIC were significantly higher i...,periodontitis,Generalised,Homo species
1,HS3ST2,Gene_Disease,DOID:4535,gene,J,disease,PharmKG,heparan sulfate-glucosamine 3-sulfotransferase 2,NCBI_ID,DOID_ID,hypotrichosis,NaN,NaN,hypotrichosis,Generalised,Homo species
2,CDH1,Gene_Disease,DOID:9296,gene,Ud,disease,PharmKG,fizzy and cell division cycle 20 related 1,NCBI_ID,DOID_ID,cleft lip,23124477.0,"'Cleft_lip , cleft_palate , hereditary_diffuse...",cleft lip,Generalised,Homo species
3,PRLR,Gene_Disease,DOID:9970,gene,Y,disease,PharmKG,prolactin receptor,NCBI_ID,DOID_ID,obesity,"24667351.0, 22637534.0",'Along this line prolactin_receptor deficient ...,obesity,Generalised,Homo species
4,EZH2,Gene_Disease,DOID:3571,gene,L,disease,PharmKG,enhancer of zeste 2 polycomb repressive comple...,NCBI_ID,DOID_ID,liver cancer,"23562851.0, 15856046.0, 24211739.0, 24570920.0...","'AIMS : The EZH2 gene , which is expressed in ...",liver cancer,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28268,DUSP6,Gene_Disease,DOID:3312,gene,J,disease,PharmKG,dual specificity phosphatase 6,NCBI_ID,DOID_ID,bipolar disorder,"22155192.0, 22155192.0, 22155192.0, 22155192.0...",'We found no significant associations with DUS...,bipolar disorder,Generalised,Homo species
28269,IL10,Gene_Disease,DOID:9065,gene,L,disease,PharmKG,interleukin 10,NCBI_ID,DOID_ID,leishmaniasis,NaN,NaN,leishmaniasis,Generalised,Homo species
28270,FOS,Gene_Disease,DOID:1686,gene,L,disease,PharmKG,"Fos proto-oncogene, AP-1 transcription factor ...",NCBI_ID,DOID_ID,glaucoma,NaN,NaN,glaucoma,Generalised,Homo species
28271,MUC2,Gene_Disease,DOID:2377,gene,L,disease,PharmKG,"mucin 2, oligomeric mucus/gel-forming",NCBI_ID,DOID_ID,multiple sclerosis,NaN,NaN,multiple sclerosis,Generalised,Homo species


### 2d. TARKG

In [21]:
TARKG_Gene_Disease = pd.read_csv(PROC_DIR + 'TARKG/Gene_Disease_TARKG.csv')
TARKG_Gene_Disease.columns = TARKG_Gene_Disease.columns.str.lower()
print(f"TARKG:       {len(TARKG_Gene_Disease):,} rows")
TARKG_Gene_Disease['kg_type'] = 'Generalised'
TARKG_Gene_Disease['kg_source'] = 'TARKG'
TARKG_Gene_Disease['species'] = 'Homo species'
TARKG_Gene_Disease.head(2)

TARKG:       57,569 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_index,kg,change,kg_type,species
0,ACE,Gene_Disease,DOID:175,Gene,GNBR::Md::Gene:Disease,Disease,TARKG,angiotensin I converting enzyme,vascular cancer,NCBI_ID,DOID,DRKG-1712794,DRKG,0,Generalised,Homo species
1,MDM2,Gene_Disease,DOID:6193,Gene,GNBR::L::Gene:Disease,Disease,TARKG,MDM2 proto-oncogene,epithelioid sarcoma,NCBI_ID,DOID,DRKG-1750804,DRKG,0,Generalised,Homo species


### 2e. Harmonizome 

In [22]:
harmonizome = pd.read_csv(PROC_DIR + f'harmonizome/Gene_Disease_Harmonizome.csv')
harmonizome.columns = harmonizome.columns.str.lower()
print(f"harmonizome:{len(harmonizome):,} rows")
harmonizome['kg_type'] = 'Generalised'
harmonizome['kg_source'] = 'Harmonizome'
harmonizome['species'] = 'Homo species'
harmonizome

harmonizome:974,298 rows


,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,VHL,Gene_Disease,DOID:5241,Gene,Disease,DISEASES Curated Gene-Disease Assocation Evide...,Harmonizome,von Hippel-Lindau tumor suppressor,hemangioblastoma,NCBI_ID,DOID,Generalised,Homo species
1,VHL,Gene_Disease,DOID:14175,Gene,Disease,DISEASES Curated Gene-Disease Assocation Evide...,Harmonizome,von Hippel-Lindau tumor suppressor,von hippel-lindau disease,NCBI_ID,DOID,Generalised,Homo species
2,VHL,Gene_Disease,DOID:255,Gene,Disease,DISEASES Curated Gene-Disease Assocation Evide...,Harmonizome,von Hippel-Lindau tumor suppressor,hemangioma,NCBI_ID,DOID,Generalised,Homo species
3,VHL,Gene_Disease,DOID:0060084,Gene,Disease,DISEASES Curated Gene-Disease Assocation Evide...,Harmonizome,von Hippel-Lindau tumor suppressor,cell type benign neoplasm,NCBI_ID,DOID,Generalised,Homo species
4,VHL,Gene_Disease,DOID:0060072,Gene,Disease,DISEASES Curated Gene-Disease Assocation Evide...,Harmonizome,von Hippel-Lindau tumor suppressor,benign neoplasm,NCBI_ID,DOID,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
974293,RWDD2A,Gene_Disease,DOID:8778,Gene,Disease,GEO,Harmonizome,RWD domain containing 2A,Crohn's disease,NCBI_ID,DOID,Generalised,Homo species
974294,ASPHD1,Gene_Disease,DOID:8778,Gene,Disease,GEO,Harmonizome,aspartate beta-hydroxylase domain containing 1,Crohn's disease,NCBI_ID,DOID,Generalised,Homo species
974295,INSL3,Gene_Disease,DOID:8778,Gene,Disease,GEO,Harmonizome,insulin like 3,Crohn's disease,NCBI_ID,DOID,Generalised,Homo species
974296,CUX1,Gene_Disease,DOID:8778,Gene,Disease,GEO,Harmonizome,cut like homeobox 1,Crohn's disease,NCBI_ID,DOID,Generalised,Homo species


In [23]:
# harmonizome[~harmonizome['tail'].str.startswith('DO')]

# keep rows where 'head' matches valid ID patterns
pattern = r'^(DOID:\d+|C\d+|D\d+)$'
harmonizome = harmonizome[harmonizome['head'].astype(str).str.match(pattern, na=False)]
harmonizome

,head,relation,tail,head_type,tail_type,source,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
1112,C3,Gene_Disease,DOID:74,Gene,Disease,DISEASES Curated Gene-Disease Assocation Evide...,Harmonizome,complement C3,hematopoietic system disease,NCBI_ID,DOID,Generalised,Homo species
1215,C3,Gene_Disease,DOID:2355,Gene,Disease,DISEASES Curated Gene-Disease Assocation Evide...,Harmonizome,complement C3,anemia,NCBI_ID,DOID,Generalised,Homo species
1284,C3,Gene_Disease,DOID:2914,Gene,Disease,DISEASES Curated Gene-Disease Assocation Evide...,Harmonizome,complement C3,immune system disease,NCBI_ID,DOID,Generalised,Homo species
1484,C3,Gene_Disease,DOID:583,Gene,Disease,DISEASES Curated Gene-Disease Assocation Evide...,Harmonizome,complement C3,hemolytic anemia,NCBI_ID,DOID,Generalised,Homo species
1519,C3,Gene_Disease,DOID:720,Gene,Disease,DISEASES Curated Gene-Disease Assocation Evide...,Harmonizome,complement C3,normocytic anemia,NCBI_ID,DOID,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
970271,C7,Gene_Disease,DOID:2377,Gene,Disease,GAD Gene-Disease Associations,Harmonizome,complement C7,multiple sclerosis,NCBI_ID,DOID,Generalised,Homo species
971398,C2,Gene_Disease,DOID:1936,Gene,Disease,GAD Gene-Disease Associations,Harmonizome,complement C2,atherosclerosis,NCBI_ID,DOID,Generalised,Homo species
971450,C5,Gene_Disease,DOID:1936,Gene,Disease,GAD Gene-Disease Associations,Harmonizome,complement C5,atherosclerosis,NCBI_ID,DOID,Generalised,Homo species
973080,C9,Gene_Disease,DOID:162,Gene,Disease,GAD High Level Gene-Disease Associations,Harmonizome,complement C9,cancer,NCBI_ID,DOID,Generalised,Homo species


In [25]:
# harmonizome[harmonizome['tail_detail_name'] == '1-Alkyl-2-acetylglycerophosphocholine Esterase']

### 2f. hald

In [26]:
hald = pd.read_csv(PROC_DIR + 'hald/Gene_Disease_HALD.csv')
hald.columns = hald.columns.str.lower()
print(f"hald:      {len(hald):,} rows")
hald['kg_type'] = 'Aging'
hald['kg_source'] = 'HALD_KG'
hald['species'] = 'Homo species'
hald

hald:      12,892 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_detail_name,tail_detail_name,head_id_is,tail_id_is,kg_type,species
0,GPT,Gene_Disease,D003643,Gene,recognized,Disease,HALD_KG,glutamic--pyruvic transaminase,Death,NCBI_ID,MESH,Aging,Homo species
1,GPT,Gene_Disease,D003643,Gene,increase,Disease,HALD_KG,glutamic--pyruvic transaminase,Death,NCBI_ID,MESH,Aging,Homo species
2,CD8A,Gene_Disease,D008175,Gene,expanded,Disease,HALD_KG,CD8 subunit alpha,Lung Neoplasms,NCBI_ID,MESH,Aging,Homo species
3,EPO,Gene_Disease,DOID:2355,Gene,delay,Disease,HALD_KG,erythropoietin,Anemia,NCBI_ID,DOID,Aging,Homo species
4,EPO,Gene_Disease,D007676,Gene,delay,Disease,HALD_KG,erythropoietin,"Kidney Failure, Chronic",NCBI_ID,MESH,Aging,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12887,PCA3,Gene_Disease,D011471,Gene,include,Disease,HALD_KG,prostate cancer associated 3,Prostatic Neoplasms,NCBI_ID,MESH,Aging,Homo species
12888,MEG3,Gene_Disease,DOID:8398,Gene,downregulated,Disease,HALD_KG,maternally expressed 3,Osteoarthritis,NCBI_ID,DOID,Aging,Homo species
12889,MIR21,Gene_Disease,D009369,Gene,reported,Disease,HALD_KG,microRNA 21,Neoplasms,NCBI_ID,MESH,Aging,Homo species
12890,TERC,Gene_Disease,D009369,Gene,exhibit,Disease,HALD_KG,telomerase RNA component,Neoplasms,NCBI_ID,MESH,Aging,Homo species


### 2g. AgeAno

In [27]:
AgeAnno = pd.read_csv(PROC_DIR + 'ageanno_processed/AgeAnno_Gene_Disease.csv')
AgeAnno.columns = AgeAnno.columns.str.lower()
print(f"AgeAnno:      {len(AgeAnno):,} rows")
AgeAnno['kg_type'] = 'Aging'
AgeAnno['kg_source'] = 'AgeAnno'
AgeAnno['species'] = 'Homo species'
AgeAnno.head(2)

AgeAnno:      590,719 rows


,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,relation_source,head_id_is,tail_id_is,species,kg_type,kg_source
0,A1BG,Gene_Disease,C0001418,Gene,Disease,alpha-1-B glycoprotein,adenocarcinoma,AgeAnno,NCBI_ID,MESH_ID,Homo species,Aging,AgeAnno
1,A1BG,Gene_Disease,C0002736,Gene,Disease,alpha-1-B glycoprotein,amyotrophic lateral sclerosis,AgeAnno,NCBI_ID,MESH_ID,Homo species,Aging,AgeAnno


In [28]:
AgeAnno[AgeAnno['tail_detail_name'].isna()]

,head,relation,tail,head_type,tail_type,head_detail_name,tail_detail_name,relation_source,head_id_is,tail_id_is,species,kg_type,kg_source
22,A1BG,Gene_Disease,C1611743,Gene,Disease,alpha-1-B glycoprotein,NaN,AgeAnno,NCBI_ID,MESH_ID,Homo species,Aging,AgeAnno
25,A1BG,Gene_Disease,C3854222,Gene,Disease,alpha-1-B glycoprotein,NaN,AgeAnno,NCBI_ID,MESH_ID,Homo species,Aging,AgeAnno
49,A2M,Gene_Disease,C0011570,Gene,Disease,alpha-2-macroglobulin,NaN,AgeAnno,NCBI_ID,MESH_ID,Homo species,Aging,AgeAnno
51,A2M,Gene_Disease,C0011847,Gene,Disease,alpha-2-macroglobulin,NaN,AgeAnno,NCBI_ID,MESH_ID,Homo species,Aging,AgeAnno
57,A2M,Gene_Disease,C0014072,Gene,Disease,alpha-2-macroglobulin,NaN,AgeAnno,NCBI_ID,MESH_ID,Homo species,Aging,AgeAnno
...,...,...,...,...,...,...,...,...,...,...,...,...,...
590689,OCLN,Gene_Disease,C4048329,Gene,Disease,occludin,NaN,AgeAnno,NCBI_ID,MESH_ID,Homo species,Aging,AgeAnno
590693,OCLN,Gene_Disease,C4529962,Gene,Disease,occludin,NaN,AgeAnno,NCBI_ID,MESH_ID,Homo species,Aging,AgeAnno
590701,OCLN,Gene_Disease,C4704874,Gene,Disease,occludin,NaN,AgeAnno,NCBI_ID,MESH_ID,Homo species,Aging,AgeAnno
590704,SLFN12L,Gene_Disease,C0037290,Gene,Disease,schlafen family member 12 like,NaN,AgeAnno,NCBI_ID,MESH_ID,Homo species,Aging,AgeAnno


## 3. Check and Fix Duplicate Columns

In [29]:
SOURCE_DFS = [
    ('Monarch_Gene_Disease', Monarch_Gene_Disease),
    ('DRKG_Gene_Disease',    DRKG_Gene_Disease),
    ('PharmKG_Gene_Disease', PharmKG_Gene_Disease),
    ('TARKG_Gene_Disease',   TARKG_Gene_Disease),
    ('harmonizome',   harmonizome),
    ('hald',   hald),
    ('AgeAnno',   AgeAnno),
]
    

for name, df in SOURCE_DFS:
    dupes = df.columns[df.columns.duplicated(keep=False)].unique().tolist()
    if dupes:
        print(f"\n[{name}] duplicate columns: {dupes}")
        for col in dupes:
            for i, (_, s) in enumerate(df.loc[:, df.columns == col].items()):
                print(f"  '{col}' occurrence {i} → sample: {s.dropna().head(3).tolist()}")
    else:
        print(f"[{name}] ✓ no duplicates")

[Monarch_Gene_Disease] ✓ no duplicates
[DRKG_Gene_Disease] ✓ no duplicates
[PharmKG_Gene_Disease] ✓ no duplicates
[TARKG_Gene_Disease] ✓ no duplicates
[harmonizome] ✓ no duplicates
[hald] ✓ no duplicates
[AgeAnno] ✓ no duplicates


In [ ]:
MESH

## 4. Align Schemas and Concatenate

In [30]:
CONCAT_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name','kg_type', 'species'
]

df_list = []
for name, df in SOURCE_DFS:
    tmp = df.loc[:, ~df.columns.duplicated()].copy()
    for col in CONCAT_COLS:
        if col not in tmp.columns:
            tmp[col] = pd.NA
    df_list.append(tmp[CONCAT_COLS])

final_df = pd.concat(df_list, ignore_index=True)
print(f"Combined: {len(final_df):,} rows")
final_df.head(2)

Combined: 766,077 rows


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,CARD9,Gene_Disease,C1859353,Gene,causes,Disease,MonarchKG,HGNC,MESH,caspase recruitment domain family member 9,predisposition to invasive fungal disease due ...,Generalised,Homo species
1,TBC1D7,Gene_Disease,C3806412,Gene,causes,Disease,MonarchKG,HGNC,MESH,TBC1 domain family member 7,"macrocephaly/megalencephaly syndrome, autosoma...",Generalised,Homo species


In [31]:
set(final_df['head_id_is']), set(final_df['tail_id_is'])

({'HGNC', 'NCBI_ID'}, {'DOID', 'DOID_ID', 'MESH', 'MESH_ID'})

In [35]:
standardize_dict = {
    'DO_ID'       : 'DOID',
    'DOID_ID'     : 'DOID',
    'MESH_ID'     : 'MESH',
}

# standardize both columns
final_df['head_id_is'] = 'NCBI_ID'
final_df['tail_id_is'] = final_df['tail_id_is'].replace(standardize_dict)
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,CARD9,Gene_Disease,C1859353,Gene,causes,Disease,MonarchKG,NCBI_ID,MESH,caspase recruitment domain family member 9,predisposition to invasive fungal disease due ...,Generalised,Homo species
1,TBC1D7,Gene_Disease,C3806412,Gene,causes,Disease,MonarchKG,NCBI_ID,MESH,TBC1 domain family member 7,"macrocephaly/megalencephaly syndrome, autosoma...",Generalised,Homo species
2,IFT81,Gene_Disease,DOID:0080295,Gene,causes,Disease,MonarchKG,NCBI_ID,DOID,intraflagellar transport 81,short-rib thoracic dysplasia 19 with or withou...,Generalised,Homo species
3,LZTR1,Gene_Disease,DOID:0060588,Gene,causes,Disease,MonarchKG,NCBI_ID,DOID,leucine zipper like post translational regulat...,Noonan syndrome 10,Generalised,Homo species
4,SLC1A1,Gene_Disease,DOID:0070093,Gene,contributes_to,Disease,MonarchKG,NCBI_ID,DOID,solute carrier family 1 member 1,schizophrenia 18,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
766072,CTXND1,Gene_Disease,C0024623,Gene,NaN,Disease,AgeAnno,NCBI_ID,MESH,cortexin domain containing 1,gastric cancer,Aging,Homo species
766073,CTXND1,Gene_Disease,C0027651,Gene,NaN,Disease,AgeAnno,NCBI_ID,MESH,cortexin domain containing 1,neoplasm,Aging,Homo species
766074,CTXND1,Gene_Disease,C0206624,Gene,NaN,Disease,AgeAnno,NCBI_ID,MESH,cortexin domain containing 1,hepatoblastoma,Aging,Homo species
766075,CTXND1,Gene_Disease,C0699791,Gene,NaN,Disease,AgeAnno,NCBI_ID,MESH,cortexin domain containing 1,gastric carcinoma,Aging,Homo species


In [36]:
set(final_df['head_id_is']), set(final_df['tail_id_is'])

({'NCBI_ID'}, {'DOID', 'MESH'})

## 5. Standardise Fixed-Value Columns

In [37]:
final_df['head_type'] = 'Gene'
final_df['tail_type'] = 'Disease'

# tail_id_is: DOID/MONDO prefixes → DOID_ID, MESH → MESH_ID
final_df['tail_id_is'] = final_df['tail_id_is'].apply(
    lambda x: 'DOID_ID' if isinstance(x, str) and x.startswith(('DOID', 'MOND')) else x
)
final_df['tail_id_is'] = final_df['tail_id_is'].replace('MESH', 'MESH_ID')

print("Unique relation:",   set(final_df['relation']))
print("Unique tail_id_is:", set(final_df['tail_id_is']))
print("Unique kg_source:",  set(final_df['kg_source']))

Unique relation: {'Gene_Disease'}
Unique tail_id_is: {'MESH_ID', 'DOID_ID'}
Unique kg_source: {'DRKG', 'TARKG', 'HALD_KG', 'Harmonizome', 'PharmKG', 'AgeAnno', 'MonarchKG'}


## 6. Repair Missing Gene Head Names

In [38]:
mask = final_df['head_detail_name'].isna()
print(f"Rows needing head_detail_name repair: {mask.sum():,}")
final_df.loc[mask, 'head'] = final_df.loc[mask, 'head'].str.replace('-', '', regex=False)
final_df.loc[mask, 'head'] = (
    final_df.loc[mask, 'head']
    .map(exploded_dict_NCBI_ALL_GENE_Syn_Dict)
    .fillna(final_df.loc[mask, 'head'])
)
mask2 = final_df['head_detail_name'].isna()
final_df.loc[mask2, 'head_detail_name'] = final_df.loc[mask2, 'head'].map(NCBI_ALL_Symb_Desc_dict)
print(f"Still missing head_detail_name: {final_df['head_detail_name'].isna().sum():,}")
final_df

Rows needing head_detail_name repair: 1,431
Still missing head_detail_name: 3


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,CARD9,Gene_Disease,C1859353,Gene,causes,Disease,MonarchKG,NCBI_ID,MESH_ID,caspase recruitment domain family member 9,predisposition to invasive fungal disease due ...,Generalised,Homo species
1,TBC1D7,Gene_Disease,C3806412,Gene,causes,Disease,MonarchKG,NCBI_ID,MESH_ID,TBC1 domain family member 7,"macrocephaly/megalencephaly syndrome, autosoma...",Generalised,Homo species
2,IFT81,Gene_Disease,DOID:0080295,Gene,causes,Disease,MonarchKG,NCBI_ID,DOID_ID,intraflagellar transport 81,short-rib thoracic dysplasia 19 with or withou...,Generalised,Homo species
3,LZTR1,Gene_Disease,DOID:0060588,Gene,causes,Disease,MonarchKG,NCBI_ID,DOID_ID,leucine zipper like post translational regulat...,Noonan syndrome 10,Generalised,Homo species
4,SLC1A1,Gene_Disease,DOID:0070093,Gene,contributes_to,Disease,MonarchKG,NCBI_ID,DOID_ID,solute carrier family 1 member 1,schizophrenia 18,Generalised,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
766072,CTXND1,Gene_Disease,C0024623,Gene,NaN,Disease,AgeAnno,NCBI_ID,MESH_ID,cortexin domain containing 1,gastric cancer,Aging,Homo species
766073,CTXND1,Gene_Disease,C0027651,Gene,NaN,Disease,AgeAnno,NCBI_ID,MESH_ID,cortexin domain containing 1,neoplasm,Aging,Homo species
766074,CTXND1,Gene_Disease,C0206624,Gene,NaN,Disease,AgeAnno,NCBI_ID,MESH_ID,cortexin domain containing 1,hepatoblastoma,Aging,Homo species
766075,CTXND1,Gene_Disease,C0699791,Gene,NaN,Disease,AgeAnno,NCBI_ID,MESH_ID,cortexin domain containing 1,gastric carcinoma,Aging,Homo species


## 7. Deduplicate

In [39]:
GROUP_COLS = ['head', 'relation', 'tail']

def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))

final_df_dedup = final_df.groupby(GROUP_COLS, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':         merge_sources,
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'kg_type': merge_sources,
    'species': 'first',
})
print(f"After deduplication: {len(final_df_dedup):,} rows")

After deduplication: 681,311 rows


## 8. QC — NaN Check

In [40]:
def nan_summary(df):
    true_nan   = df.isna().sum()
    string_nan = df.apply(lambda c: c.astype(str).str.upper().eq('NAN').sum())
    return pd.DataFrame({
        'NaN_count':          true_nan,
        "'NAN'_string_count": string_nan,
        'Total_NaN_like':     true_nan + string_nan
    })

nan_summary(final_df_dedup)

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,535433,0,535433
tail_type,0,0,0
kg_source,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0
head_detail_name,3,0,3


## 9. Drop Unresolvable Rows and Add Schema Columns

In [41]:
before = len(final_df_dedup)
final_df_dedup = final_df_dedup[~final_df_dedup['head_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df_dedup):,} rows with missing head_detail_name")

final_df_dedup = final_df_dedup[REQUIRED_COLS]
print(f"Final shape: {final_df_dedup.shape}")
final_df_dedup.head()

Dropped 3 rows with missing head_detail_name
Final shape: (681308, 13)


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,6PGD,Gene_Disease,DOID:1240,Gene,L,Disease,PharmKG,NCBI_ID,DOID_ID,phosphogluconate dehydrogenase,leukemia,Generalised,Homo species
1,A1BG,Gene_Disease,C0001418,Gene,None,Disease,AgeAnno,NCBI_ID,MESH_ID,alpha-1-B glycoprotein,adenocarcinoma,Aging,Homo species
2,A1BG,Gene_Disease,C0002736,Gene,None,Disease,AgeAnno,NCBI_ID,MESH_ID,alpha-1-B glycoprotein,amyotrophic lateral sclerosis,Aging,Homo species
3,A1BG,Gene_Disease,C0003578,Gene,None,Disease,AgeAnno,NCBI_ID,MESH_ID,alpha-1-B glycoprotein,apnea,Aging,Homo species
4,A1BG,Gene_Disease,C0003864,Gene,None,Disease,AgeAnno,NCBI_ID,MESH_ID,alpha-1-B glycoprotein,arthritis,Aging,Homo species


In [42]:
final_df_dedup['tail_detail_name'] = (
    final_df_dedup['tail_detail_name']
    .fillna(
        final_df_dedup['tail'].map(MESH_dict)
        .fillna(
            final_df_dedup['tail'].map(Mesh_supp_dict)
            .fillna(
                final_df_dedup['tail'].map(rest_mesh_dict))
                .fillna(
                    final_df_dedup['tail'].map(MedGen_CUID_Source_ID_name_dict))
        )
    )
)
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,6PGD,Gene_Disease,DOID:1240,Gene,L,Disease,PharmKG,NCBI_ID,DOID_ID,phosphogluconate dehydrogenase,leukemia,Generalised,Homo species
1,A1BG,Gene_Disease,C0001418,Gene,None,Disease,AgeAnno,NCBI_ID,MESH_ID,alpha-1-B glycoprotein,adenocarcinoma,Aging,Homo species
2,A1BG,Gene_Disease,C0002736,Gene,None,Disease,AgeAnno,NCBI_ID,MESH_ID,alpha-1-B glycoprotein,amyotrophic lateral sclerosis,Aging,Homo species
3,A1BG,Gene_Disease,C0003578,Gene,None,Disease,AgeAnno,NCBI_ID,MESH_ID,alpha-1-B glycoprotein,apnea,Aging,Homo species
4,A1BG,Gene_Disease,C0003864,Gene,None,Disease,AgeAnno,NCBI_ID,MESH_ID,alpha-1-B glycoprotein,arthritis,Aging,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
681303,ZYX,Gene_Disease,C2239176,Gene,None,Disease,AgeAnno,NCBI_ID,MESH_ID,zyxin,hepatocellular carcinoma,Aging,Homo species
681304,ZYX,Gene_Disease,D009369,Gene,GNBR::L::Gene:Disease,Disease,DRKG,NCBI_ID,MESH_ID,zyxin,Neoplasms,Generalised,Homo species
681305,ZYX,Gene_Disease,D015179,Gene,GNBR::L::Gene:Disease,Disease,DRKG,NCBI_ID,MESH_ID,zyxin,Colorectal Neoplasms,Generalised,Homo species
681306,ZYX,Gene_Disease,DOID:162,Gene,GNBR::L::Gene:Disease,Disease,TARKG,NCBI_ID,DOID_ID,zyxin,cancer,Generalised,Homo species


In [43]:
final_df_dedup = final_df_dedup[~final_df_dedup['tail_detail_name'].isna()]
final_df_dedup

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,6PGD,Gene_Disease,DOID:1240,Gene,L,Disease,PharmKG,NCBI_ID,DOID_ID,phosphogluconate dehydrogenase,leukemia,Generalised,Homo species
1,A1BG,Gene_Disease,C0001418,Gene,None,Disease,AgeAnno,NCBI_ID,MESH_ID,alpha-1-B glycoprotein,adenocarcinoma,Aging,Homo species
2,A1BG,Gene_Disease,C0002736,Gene,None,Disease,AgeAnno,NCBI_ID,MESH_ID,alpha-1-B glycoprotein,amyotrophic lateral sclerosis,Aging,Homo species
3,A1BG,Gene_Disease,C0003578,Gene,None,Disease,AgeAnno,NCBI_ID,MESH_ID,alpha-1-B glycoprotein,apnea,Aging,Homo species
4,A1BG,Gene_Disease,C0003864,Gene,None,Disease,AgeAnno,NCBI_ID,MESH_ID,alpha-1-B glycoprotein,arthritis,Aging,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
681303,ZYX,Gene_Disease,C2239176,Gene,None,Disease,AgeAnno,NCBI_ID,MESH_ID,zyxin,hepatocellular carcinoma,Aging,Homo species
681304,ZYX,Gene_Disease,D009369,Gene,GNBR::L::Gene:Disease,Disease,DRKG,NCBI_ID,MESH_ID,zyxin,Neoplasms,Generalised,Homo species
681305,ZYX,Gene_Disease,D015179,Gene,GNBR::L::Gene:Disease,Disease,DRKG,NCBI_ID,MESH_ID,zyxin,Colorectal Neoplasms,Generalised,Homo species
681306,ZYX,Gene_Disease,DOID:162,Gene,GNBR::L::Gene:Disease,Disease,TARKG,NCBI_ID,DOID_ID,zyxin,cancer,Generalised,Homo species


In [44]:
final_df_dedup[final_df_dedup['tail_detail_name'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species


## 10. Save Output

In [45]:
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 645,661 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/GENE_DISEASE/ALL_GENE_DISEASE.csv


In [46]:
#

# Final

In [4]:
# =========================================================
# LOAD DISEASE MAPPING FILE
# =========================================================

DB_DIR = BASE_DIR + 'data_collection/databases_for_mapping/'

final_file_CH_D = pd.read_csv(
    DB_DIR + 'DO/ultimate_DISEASE_dictionary.csv'
)

# =========================================================
# PRIORITIZE DOID IDs
# =========================================================

# rows having DOID
doid_rows = final_file_CH_D[
    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)
]

# entity names that have at least one DOID
names_with_doid = set(
    doid_rows['entity_name']
    .str.lower()
    .str.strip()
)

# keep:
# 1. all DOID rows
# 2. non-DOID rows only if no DOID exists

final_diseaseid_df = final_file_CH_D[

    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)

    |

    ~final_file_CH_D['entity_name']
    .str.lower()
    .str.strip()
    .isin(names_with_doid)
]

final_diseaseid_df = (
    final_diseaseid_df
    .reset_index(drop=True)
)

# =========================================================
# CREATE DISEASE NAME -> ID DICTIONARY
# =========================================================

disease_dict = dict(

    zip(

        final_diseaseid_df['entity_name']
        .str.lower()
        .str.strip(),

        final_diseaseid_df['entity_id']
    )
)

# =========================================================
# UNIVERSAL ENTITY MAPPING FUNCTION
# =========================================================

def map_entity_ids(
    df,
    mapping_dict,
    side='tail'
):
    """
    Universal KG entity mapper.

    Parameters
    ----------
    df : pd.DataFrame

    mapping_dict : dict
        entity_name -> entity_id

    side : str
        'head' or 'tail'

    Returns
    -------
    pd.DataFrame
    """

    # -----------------------------------------
    # dynamic column names
    # -----------------------------------------

    detail_col = f'{side}_detail_name'

    id_col = side

    id_is_col = f'{side}_id_is'

    # -----------------------------------------
    # map IDs
    # -----------------------------------------

    df[id_col] = (df[detail_col].astype(str).str.lower().str.strip().map(mapping_dict))

    # -----------------------------------------
    # assign ID source
    # -----------------------------------------

    df[id_is_col] = None

    # DOID
    # DOID
    df.loc[
        df[id_col].str.startswith('DOID:', na=False),id_is_col] = 'DOID'
    
    # everything else -> MESH
    df.loc[~df[id_col].str.startswith('DOID:', na=False)&df[id_col].notna(),id_is_col] = 'MESH'

    return df

In [5]:
final_file = pd.read_csv(OUT_PATH)

final_file = map_entity_ids(df=final_file,mapping_dict=disease_dict,side='tail')

final_file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species
0,6PGD,Gene_Disease,DOID:1240,Gene,L,Disease,PharmKG,NCBI_ID,DOID,phosphogluconate dehydrogenase,leukemia,Generalised,Homo species
1,A1BG,Gene_Disease,DOID:299,Gene,NaN,Disease,AgeAnno,NCBI_ID,DOID,alpha-1-B glycoprotein,adenocarcinoma,Aging,Homo species
2,A1BG,Gene_Disease,DOID:332,Gene,NaN,Disease,AgeAnno,NCBI_ID,DOID,alpha-1-B glycoprotein,amyotrophic lateral sclerosis,Aging,Homo species
3,A1BG,Gene_Disease,C0003578,Gene,NaN,Disease,AgeAnno,NCBI_ID,MESH,alpha-1-B glycoprotein,apnea,Aging,Homo species
4,A1BG,Gene_Disease,DOID:848,Gene,NaN,Disease,AgeAnno,NCBI_ID,DOID,alpha-1-B glycoprotein,arthritis,Aging,Homo species
...,...,...,...,...,...,...,...,...,...,...,...,...,...
645656,ZYX,Gene_Disease,DOID:684,Gene,NaN,Disease,AgeAnno,NCBI_ID,DOID,zyxin,hepatocellular carcinoma,Aging,Homo species
645657,ZYX,Gene_Disease,D009369,Gene,GNBR::L::Gene:Disease,Disease,DRKG,NCBI_ID,MESH,zyxin,Neoplasms,Generalised,Homo species
645658,ZYX,Gene_Disease,D015179,Gene,GNBR::L::Gene:Disease,Disease,DRKG,NCBI_ID,MESH,zyxin,Colorectal Neoplasms,Generalised,Homo species
645659,ZYX,Gene_Disease,DOID:162,Gene,GNBR::L::Gene:Disease,Disease,TARKG,NCBI_ID,DOID,zyxin,cancer,Generalised,Homo species


In [6]:
final_file[final_file['tail'].isna()]


,head,relation,tail,head_type,relation_type,tail_type,kg_source,head_id_is,tail_id_is,head_detail_name,tail_detail_name,kg_type,species


In [7]:
final_file.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_file):,} rows → {OUT_PATH}")

Saved 645,661 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/GENE_DISEASE/ALL_GENE_DISEASE.csv
